# 05 — Drug Candidate Generation & Scoring
**Generate → Score → Filter → Rank**

This notebook uses the optimized model to evaluate drug candidates generated via 3 strategies:

1. **Scaffold Decoration** — take known EGFR scaffolds, vary substituents
2. **SMILES Mutation** — random valid perturbations of active compounds
3. **Fragment Assembly** — combine known pharmacophore fragments

All candidates are:
- Scored by the optimized model (predicted IC50)
- Filtered by Lipinski Rule of Five
- Ranked by predicted activity + drug-likeness
- Compared against known EGFR drugs

> **Prerequisite:** `04_optimize.ipynb` must have run.

## 0. Imports & Setup

In [ ]:
import sys, json, warnings
sys.path.insert(0, '.')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Lipinski
from rdkit.Chem import rdMolDescriptors
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from utils.mol_utils import passes_lipinski, filter_lipinski

with open('./config.json') as f:
    CONFIG = json.load(f)

with open('./data/known_drugs.json') as f:
    KNOWN_DRUGS = json.load(f)

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = CONFIG['use_amp'] and torch.cuda.is_available()
WINNER  = CONFIG.get('winner_model', 'chemberta2')

print(f'Device: {DEVICE}')
print(f'Scoring model: {WINNER}')
print(f'Known reference drugs: {list(KNOWN_DRUGS.keys())}')

## 1. Load Optimized Scoring Model

In [ ]:
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from torch.cuda.amp import autocast

MODEL_NAME = 'seyonec/ChemBERTa-zinc-base-v1'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

class ChemBERTaRegressor(nn.Module):
    def __init__(self, model_name, hidden_size=256, dropout=0.2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        bert_hidden = self.bert.config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(bert_hidden, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.regressor(cls).squeeze(-1)

scoring_model = ChemBERTaRegressor(MODEL_NAME).to(DEVICE)
scoring_model.load_state_dict(torch.load('./models/chemberta2_best.pt', map_location=DEVICE))
scoring_model.eval()
print('Scoring model loaded.')

def score_smiles(smiles_list: list, batch_size: int = 64) -> np.ndarray:
    """Score a list of SMILES strings. Returns predicted log(IC50)."""
    all_preds = []
    for i in range(0, len(smiles_list), batch_size):
        batch = smiles_list[i:i+batch_size]
        enc = tokenizer(batch, truncation=True, padding='max_length',
                        max_length=128, return_tensors='pt')
        with torch.no_grad():
            with autocast(enabled=USE_AMP):
                preds = scoring_model(
                    enc['input_ids'].to(DEVICE),
                    enc['attention_mask'].to(DEVICE)
                )
        all_preds.extend(preds.cpu().float().numpy())
    return np.array(all_preds)

## 2. STRATEGY A — Scaffold Decoration
Takes a core EGFR scaffold and varies the R-groups at known attachment points.

In [ ]:
from rdkit.Chem import AllChem

print('Strategy A: Scaffold Decoration')
print('='*45)

# Core 4-anilinoquinazoline scaffold (shared by gefitinib, erlotinib, afatinib)
EGFR_SCAFFOLDS = [
    'c1ccc2c(N)ncnc2c1',                          # Quinazoline core
    'c1cnc2ccccc2n1',                              # Quinoxaline core
    'c1ccc2[nH]nnc2c1',                           # Indazole core
]

# R-group libraries at each position
R_GROUPS_ANILINE = [
    'Fc1ccc(Cl)cc1',        # 3-Cl-4-F-aniline (gefitinib-like)
    'Fc1ccc(Br)cc1',        # fluorobromobenzene
    'Clc1ccccc1',           # chlorobenzene
    'c1ccc(F)cc1',          # fluorobenzene
    'c1ccncc1',             # pyridine
    'C1CCNCC1',             # piperidine
    'c1ccoc1',              # furan
    'c1ccsc1',              # thiophene
    'Cc1ccccc1',            # toluene
    'COc1ccccc1',           # anisole
    'FC(F)(F)c1ccccc1',     # trifluoromethylbenzene
]

R_GROUPS_TAIL = [
    'OCC',                  # ethanol
    'OCCO',                 # ethanediol
    'OCc1cccnc1',           # pyridylmethanol
    'CN(C)CCO',             # dimethylaminoethanol
    'N1CCOCC1',             # morpholine
    'N1CCNCC1',             # piperazine
    'OC(=O)C',              # acetic acid
    'C#N',                  # nitrile
    'NC(=O)C',              # acetamide
    'S(=O)(=O)N',           # sulfonamide
]

def generate_scaffold_variants(scaffold_smi, r_groups, max_variants=200):
    """Generate variants by attaching R-groups to scaffold attachment points."""
    candidates = []
    scaffold = Chem.MolFromSmiles(scaffold_smi)
    if scaffold is None:
        return candidates

    for rg in r_groups:
        # Simple concatenation via SMILES — real attachment uses reaction SMARTS
        # This is a simplified but chemically reasonable approach for prototyping
        combined = f'{scaffold_smi}.{rg}'
        mol = Chem.MolFromSmiles(combined)
        if mol is not None:
            canonical = Chem.MolToSmiles(mol)
            candidates.append(canonical)

    # Also try direct scaffold-rgroup linking
    for rg_a in r_groups[:5]:
        for rg_t in r_groups[:5]:
            smi = f'{scaffold_smi}N{rg_a}'
            mol = Chem.MolFromSmiles(smi)
            if mol is not None:
                candidates.append(Chem.MolToSmiles(mol))
            if len(candidates) >= max_variants:
                break

    return list(set(candidates))[:max_variants]

strategy_a = []
for scaffold in EGFR_SCAFFOLDS:
    variants = generate_scaffold_variants(scaffold, R_GROUPS_ANILINE + R_GROUPS_TAIL)
    strategy_a.extend(variants)

strategy_a = list(set(strategy_a))
print(f'  Generated {len(strategy_a)} candidates via scaffold decoration')

## 3. STRATEGY B — SMILES Mutation
Randomly perturbs highly active compounds from the training set.

In [ ]:
import random
random.seed(42)

print('Strategy B: SMILES Mutation')
print('='*45)

df_train = pd.read_csv('./data/train.csv')

# Start from the top 5% most active compounds
threshold = df_train['Y_log'].quantile(0.05)  # low IC50 = high activity
actives = df_train[df_train['Y_log'] < threshold]['Drug'].tolist()
print(f'  Seed compounds (top 5% active): {len(actives)}')

ATOM_SWAPS = {
    'C': ['N', 'O', 'S'],
    'N': ['C', 'O', 'S'],
    'O': ['N', 'S'],
    'S': ['O', 'N'],
    'F': ['Cl', 'Br'],
    'Cl': ['F', 'Br'],
    'Br': ['F', 'Cl'],
}

def mutate_smiles(smiles: str, n_mutations: int = 1) -> list:
    """Apply random single-atom swaps to produce valid mutant SMILES."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return []

    mutants = []
    atoms = [a for a in mol.GetAtoms() if a.GetSymbol() in ATOM_SWAPS]
    if not atoms:
        return []

    for _ in range(n_mutations * 3):  # Try more than needed to get valid ones
        atom = random.choice(atoms)
        swap = random.choice(ATOM_SWAPS[atom.GetSymbol()])

        # Build new SMILES with swap
        original = Chem.MolToSmiles(mol)
        mutated_smi = original.replace(atom.GetSymbol(), swap, 1)
        mutated_mol = Chem.MolFromSmiles(mutated_smi)

        if mutated_mol is not None:
            canonical = Chem.MolToSmiles(mutated_mol)
            if canonical != original:
                mutants.append(canonical)

    return mutants[:n_mutations]

strategy_b = []
for smi in tqdm(actives[:200], desc='Mutating'):
    mutants = mutate_smiles(smi, n_mutations=5)
    strategy_b.extend(mutants)

strategy_b = list(set(strategy_b))
print(f'  Generated {len(strategy_b)} candidates via SMILES mutation')

## 4. STRATEGY C — Fragment Assembly
Combines known pharmacophore fragments from EGFR inhibitor literature.

In [ ]:
print('Strategy C: Fragment Assembly')
print('='*45)

# Known pharmacophore fragments from EGFR inhibitors
# Based on: hinge binder, linker, solubilizing group
HINGE_BINDERS = [
    'Nc1ncnc2ccccc12',       # 4-aminoquinazoline
    'Nc1ncnc2[nH]ncc12',     # 4-aminopyrrolo-pyrimidine
    'Nc1nccc2ccccc12',       # 4-aminoquinoline
    'Nc1ccnc2ccccc12',       # aminoisoquinoline
]

LINKERS = [
    'NCC',
    'NCCO',
    'NC(=O)',
    'OCC',
    'NCS',
]

WARHEADS = [  # Covalent warheads for irreversible inhibitors (like afatinib)
    'C=CC(=O)',              # Acrylamide — Michael acceptor
    'C#CC(=O)',              # Propiolamide
    'ClCC(=O)',              # Chloroacetamide
]

SOLUBILIZERS = [
    'N1CCOCC1',              # Morpholine
    'N1CCNCC1',              # Piperazine
    'N1CCCC1',               # Pyrrolidine
    'CN(C)C',                # Dimethylamine
    'OCC(O)',                # Diol
]

strategy_c = []

for hinge in HINGE_BINDERS:
    for linker in LINKERS:
        for sol in SOLUBILIZERS:
            # Assemble fragment combination
            assembled = f'{hinge}{linker}{sol}'
            mol = Chem.MolFromSmiles(assembled)
            if mol is not None:
                strategy_c.append(Chem.MolToSmiles(mol))

        for wh in WARHEADS:  # Also try with warhead
            assembled = f'{hinge}{linker}{wh}'
            mol = Chem.MolFromSmiles(assembled)
            if mol is not None:
                strategy_c.append(Chem.MolToSmiles(mol))

strategy_c = list(set(strategy_c))
print(f'  Generated {len(strategy_c)} candidates via fragment assembly')

## 5. Score & Filter All Candidates

In [ ]:
# Combine all candidates
all_candidates = {
    'scaffold_decoration': strategy_a,
    'smiles_mutation':     strategy_b,
    'fragment_assembly':   strategy_c,
}

records = []
for strategy, smiles_list in all_candidates.items():
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        records.append({'smiles': smi, 'strategy': strategy})

df_cands = pd.DataFrame(records).drop_duplicates('smiles').reset_index(drop=True)
print(f'Total unique candidates: {len(df_cands):,}')
print()

# ── Score with optimized model ─────────────────────────────────
print('Scoring all candidates...')
scores = score_smiles(df_cands['smiles'].tolist())
df_cands['pred_log_ic50'] = scores
df_cands['pred_ic50_nm']  = np.expm1(scores)  # Reverse log transform

# ── Lipinski filter ────────────────────────────────────────────
print('Applying Lipinski Rule of Five filter...')
df_cands['passes_ro5'] = df_cands['smiles'].apply(passes_lipinski)

# ── Compute molecular descriptors ─────────────────────────────
def get_props(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return {}
    return {
        'mol_weight': Descriptors.MolWt(mol),
        'logp':       Descriptors.MolLogP(mol),
        'hbd':        Lipinski.NumHDonors(mol),
        'hba':        Lipinski.NumHAcceptors(mol),
        'tpsa':       Descriptors.TPSA(mol),
        'rot_bonds':  Lipinski.NumRotatableBonds(mol),
    }

props = df_cands['smiles'].apply(get_props).apply(pd.Series)
df_cands = pd.concat([df_cands, props], axis=1)

n_before = len(df_cands)
df_filtered = df_cands[df_cands['passes_ro5']].copy()
n_after = len(df_filtered)

print(f'\nLipinski filter:')
print(f'  Before: {n_before:,} candidates')
print(f'  After:  {n_after:,} candidates ({n_after/n_before*100:.0f}% pass Ro5)')

## 6. Rank & Display Top Candidates

In [ ]:
# Score reference drugs for comparison
ref_scores = score_smiles(list(KNOWN_DRUGS.values()))
df_ref = pd.DataFrame({
    'name':           list(KNOWN_DRUGS.keys()),
    'smiles':         list(KNOWN_DRUGS.values()),
    'pred_log_ic50':  ref_scores,
    'pred_ic50_nm':   np.expm1(ref_scores),
    'is_reference':   True
})

# Rank candidates (lower pred IC50 = more potent)
df_top = df_filtered.sort_values('pred_log_ic50', ascending=True).head(50).copy()
df_top['rank'] = range(1, len(df_top) + 1)

print('TOP 20 DRUG CANDIDATES')
print('='*75)
display_cols = ['rank', 'smiles', 'strategy', 'pred_ic50_nm', 'mol_weight', 'logp']
print(df_top[display_cols].head(20).to_string(index=False))

print('\nREFERENCE DRUGS (for comparison):')
print('-'*50)
for _, row in df_ref.iterrows():
    print(f"  {row['name']:<15} pred IC50: {row['pred_ic50_nm']:.1f} nM")

# Save results
df_top.to_csv('./results/top_candidates.csv', index=False)
df_ref.to_csv('./results/reference_scores.csv', index=False)
print('\nSaved to results/top_candidates.csv')

## 7. Final Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Generated Drug Candidates — EGFR Kinase', fontsize=13, fontweight='bold')

STRAT_COLORS = {
    'scaffold_decoration': '#5C85D6',
    'smiles_mutation':     '#59B87A',
    'fragment_assembly':   '#E8874A',
}

# ── Predicted IC50 Distribution per strategy ──────────────────────
ax = axes[0]
for strat, color in STRAT_COLORS.items():
    subset = df_filtered[df_filtered['strategy'] == strat]['pred_log_ic50']
    ax.hist(subset, bins=30, alpha=0.6, color=color, label=strat.replace('_', ' '))

# Reference drug scores
for _, row in df_ref.iterrows():
    ax.axvline(row['pred_log_ic50'], color='navy', linestyle='--', alpha=0.7)
    ax.text(row['pred_log_ic50'], ax.get_ylim()[1]*0.9, row['name'],
            rotation=90, fontsize=7, color='navy')

ax.set_xlabel('Predicted log(IC50)')
ax.set_ylabel('Count')
ax.set_title('Activity Distribution by Strategy')
ax.legend(fontsize=8)

# ── Top 20 candidates ranked ──────────────────────────────────────
ax = axes[1]
top20 = df_top.head(20)
bar_colors = [STRAT_COLORS.get(s, '#888') for s in top20['strategy']]
bars = ax.barh(range(len(top20)), top20['pred_log_ic50'], color=bar_colors)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels([f'#{i+1}' for i in range(len(top20))], fontsize=8)
ax.set_xlabel('Predicted log(IC50) — lower = more potent')
ax.set_title('Top 20 Candidates')
ax.invert_yaxis()

# Reference lines
for _, row in df_ref.iterrows():
    ax.axvline(row['pred_log_ic50'], color='navy', linestyle=':', alpha=0.6)

patches = [mpatches.Patch(color=c, label=k.replace('_', ' '))
           for k, c in STRAT_COLORS.items()]
ax.legend(handles=patches, fontsize=7, loc='lower right')

# ── MW vs LogP with activity coloring ────────────────────────────
ax = axes[2]
sc = ax.scatter(df_filtered['mol_weight'], df_filtered['logp'],
               c=df_filtered['pred_log_ic50'], cmap='RdYlGn_r',
               alpha=0.5, s=8)
plt.colorbar(sc, ax=ax, label='Predicted log(IC50)')

# Lipinski region
ax.axvline(x=500, color='red', linestyle='--', alpha=0.5, label='MW=500 (Ro5)')
ax.axhline(y=5,   color='orange', linestyle='--', alpha=0.5, label='LogP=5 (Ro5)')

# Reference drugs
for _, row in df_ref.iterrows():
    mol = Chem.MolFromSmiles(row['smiles'])
    if mol:
        mw = Descriptors.MolWt(mol)
        lp = Descriptors.MolLogP(mol)
        ax.scatter(mw, lp, s=120, marker='*', color='blue', zorder=5)
        ax.annotate(row['name'], (mw, lp), fontsize=7, color='blue',
                    textcoords='offset points', xytext=(4, 4))

ax.set_xlabel('Molecular Weight (Da)')
ax.set_ylabel('LogP')
ax.set_title('Chemical Space (★ = known drugs)')
ax.legend(fontsize=7)

import matplotlib.patches as mpatches

plt.tight_layout()
plt.savefig('./results/05_candidates.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to results/05_candidates.png')

## ✓ Pipeline Complete

**What we built:**

| Notebook | Output |
|---|---|
| `00_setup` | RTX 3070 verified, AMP configured, CodeCarbon ready |
| `01_data`  | EGFR dataset cleaned, split, EDA |
| `02_baselines` | 4 models trained with accuracy + VRAM + CO2 tracked |
| `03_comparison` | Benchmark table, efficiency scores, winner selected |
| `04_optimize` | Gradient checkpointing, distillation, INT8 quantization, Optuna |
| `05_generate` | Drug candidates generated, scored, filtered, ranked |

**Next steps:**
- Wrap this pipeline in a Streamlit app for interactive candidate exploration
- Add Gnina/AutoDock for physics-based docking validation
- Connect to ADMET prediction for full drug-likeness scoring
- Compare final model R² against GPT-Rosalind BixBench scores (when published)